# 🥣 Kohlenhydrat-Rechner (USDA-Variante, ohne LLM)

Diese Variante nutzt **keine** KI zur Schätzung, sondern die [USDA FoodData Central API](https://fdc.nal.usda.gov/) — eine öffentliche, kostenlose Nährwertdatenbank der US-Regierung.

**Wichtiger Unterschied zur LLM-Variante:** Da kein Modell mehr die Zutatengewichte *schätzt*, musst du sie jetzt **explizit in Gramm** angeben.

Beispiel-Eingabe:
```
150 g Haferflocken, 50 g Apfel, 60 g Banane, 75 g Pfirsich
```

## Architektur (DDD)

Die Restaurant-Analogie bleibt, ändert sich aber an einer Stelle: der Kellner muss nicht mehr *schätzen*, wie viel auf dem Teller liegt — das steht jetzt auf der Bestellung. Er muss nur noch im **Nährwert-Nachschlagewerk** (USDA) blättern.

| DDD-Begriff | Hier konkret | Restaurant-Analogie |
|---|---|---|
| Value Object | `Carbohydrates`, `Ingredient` | Posten auf der Rechnung |
| Aggregate Root | `Meal` | Die ganze Bestellung |
| Domain Service (Parsing) | `IngredientParser` | Kellner liest die Bestellung |
| Domain Service (Orchestrierung) | `MealAnalyzer` | Kellner reicht Bestellung weiter |
| Infrastructure | `USDANutritionRepository` | Das Nährwert-Nachschlagewerk (USDA) |

Kein LLM-Aufruf mehr nötig — Parsing ist jetzt deterministisch (Regex), weil das Format der Eingabe (Zahl + Einheit + Name) fest vorgegeben ist.

## 1. Setup

### API-Key besorgen
Kostenlos unter https://fdc.nal.usda.gov/api-key-signup.html registrieren. Ohne eigenen Key funktioniert `DEMO_KEY`, ist aber **stark ratenlimitiert** (nur für kurze Tests geeignet).

In [ ]:
# %pip install requests

In [1]:
import os
import re
import requests
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from dotenv import load_dotenv

# API-Key setzen (empfohlen), sonst Fallback auf DEMO_KEY
load_dotenv()  # .env-Datei laden, falls vorhanden
# os.environ['USDA_API_KEY'] = '...'
USDA_API_KEY = os.environ.get("USDA_API_KEY", "DEMO_KEY")

if USDA_API_KEY == "DEMO_KEY":
    print("⚠️  DEMO_KEY wird verwendet — stark ratenlimitiert. "
          "Für produktiven Gebrauch einen eigenen Key setzen (USDA_API_KEY).")

## 2. Domain Model

Unverändert gegenüber den anderen Varianten — reine Datenstrukturen, keine Abhängigkeiten.

In [2]:
@dataclass(frozen=True)
class Carbohydrates:
    """Wertobjekt: Kohlenhydratmenge in Gramm. Unveränderlich, addierbar."""
    grams: float

    def __add__(self, other: "Carbohydrates") -> "Carbohydrates":
        return Carbohydrates(self.grams + other.grams)

    def __radd__(self, other):
        if other == 0:
            return self
        return self.__add__(other)

    def __str__(self) -> str:
        return f"{self.grams:.1f} g"


@dataclass(frozen=True)
class Ingredient:
    """Wertobjekt: eine einzelne Zutat mit Gewicht und Kohlenhydraten."""
    name: str                 # wie vom Benutzer eingegeben
    matched_usda_name: str    # der von USDA gefundene Treffer (Transparenz!)
    weight_g: float
    carbs: Carbohydrates


@dataclass
class Meal:
    """Aggregate Root: eine Mahlzeit besteht aus Zutaten."""
    description: str
    ingredients: List[Ingredient] = field(default_factory=list)

    @property
    def total_carbs(self) -> Carbohydrates:
        return sum(i.carbs for i in self.ingredients) or Carbohydrates(0)

    def report(self) -> str:
        lines = [f"Mahlzeit: {self.description}", "-" * 65]
        for ing in self.ingredients:
            lines.append(
                f"  • {ing.name:<20} {ing.weight_g:>6.1f} g "
                f"→  {ing.carbs}   (USDA: {ing.matched_usda_name})"
            )
        lines.append("-" * 65)
        lines.append(f"  Gesamt-Kohlenhydrate: {self.total_carbs}")
        return "\n".join(lines)

## 3. Parsing: `IngredientParser`

Kein LLM nötig, weil das Eingabeformat fest ist: `<Zahl> g <Zutat>` (Komma-getrennte Liste). Ein einfacher Regex genügt — deterministisch und kostenlos.

In [3]:
class IngredientParsingError(ValueError):
    """Wird geworfen, wenn eine Zutatenzeile kein Gewicht enthält."""


class IngredientParser:
    """Domain Service: zerlegt Freitext in (Name, Gramm)-Paare.

    Erwartetes Format pro Eintrag (Komma-getrennt): 'ZAHL g NAME' oder 'NAME ZAHL g'.
    Beispiel: '150 g Haferflocken, Apfel 50g, 60g Banane'
    """

    _WEIGHT_PATTERN = re.compile(
        r"(\d+(?:[.,]\d+)?)\s*(?:g|gramm|gr)\b\.?", re.IGNORECASE
    )

    def parse(self, text: str) -> List[Tuple[str, float]]:
        items = [part.strip() for part in text.split(",") if part.strip()]
        return [self._parse_line(item) for item in items]

    def _parse_line(self, line: str) -> Tuple[str, float]:
        match = self._WEIGHT_PATTERN.search(line)
        if not match:
            raise IngredientParsingError(
                f"Kein Gewicht in Gramm gefunden in: '{line}'. "
                f"Erwartet wird z. B. '150 g Haferflocken'."
            )
        grams = float(match.group(1).replace(",", "."))
        name = (line[: match.start()] + " " + line[match.end() :]).strip()
        name = re.sub(r"\s+", " ", name).strip(" ,.-")
        if not name:
            raise IngredientParsingError(f"Kein Zutatenname gefunden in: '{line}'")
        return name, grams

In [4]:
# Kurzer Test des Parsers (ohne API-Aufruf)
parser = IngredientParser()
test_text = "150 g Haferflocken, 50g Apfel, Banane 60g, 75 gramm Pfirsich"
print(parser.parse(test_text))

[('Haferflocken', 150.0), ('Apfel', 50.0), ('Banane', 60.0), ('Pfirsich', 75.0)]


## 4. Infrastructure: `USDANutritionRepository`

Kapselt den Zugriff auf die USDA-API. Sucht die Zutat und liest den Nährwert **Carbohydrate, by difference** (USDA-Nutrient-ID `1005`) aus. Der Wert ist bei `Foundation`/`SR Legacy`-Einträgen immer **pro 100 g**.

### Warum nicht einfach der erste Suchtreffer?

Die USDA-Relevanzsuche gewichtet reine Textähnlichkeit über die *gesamte* Beschreibung — nicht "roh vs. verarbeitet". Eine Suche nach `apple` liefert daher munter `Croissants, apple` oder `Pie, peach` gleichrangig neben `Apples, raw, with skin`. Auch `SR Legacy` enthält massenhaft verarbeitete/zusammengesetzte Gerichte, nicht nur Rohkost.

Deshalb holt das Repository jetzt **mehrere Kandidaten** und wählt danach selbst den passendsten aus, anhand einer einfachen Heuristik:

1. **Namensähnlichkeit**: Beginnt die Beschreibung (vor dem ersten Komma) ähnlich wie der gesuchte Begriff? ("Apples, raw..." passt zu "apple")
2. **Enthält "raw"**: rohe/generische Form wird bevorzugt.
3. **Straf-Wörter**: Begriffe wie *pie, croissant, dehydrated, powder, juice, dried, canned, syrup, cooked* etc. senken den Score, weil sie auf eine verarbeitete Variante hindeuten.
4. **Kürzere Beschreibung** als Tie-Breaker (generische Einträge haben meist kürzere Namen als Spezialprodukte).

Das ist eine **Heuristik, kein Beweis** — bei ungewöhnlichen Zutaten lohnt sich ein Blick auf `matched_usda_name` im Ergebnis, bzw. die Debug-Zelle weiter unten.

In [5]:
class USDALookupError(ValueError):
    """Wird geworfen, wenn keine passende Zutat oder kein Kohlenhydratwert gefunden wird."""


class USDANutritionRepository:
    """Infrastructure: liefert Kohlenhydrate pro 100 g über die USDA FoodData Central API.

    Wählt unter mehreren Suchtreffern per Heuristik den plausibelsten roh/generischen
    Eintrag aus, statt blind den ersten Treffer zu nehmen (siehe Markdown-Zelle oben).
    """

    BASE_URL = "https://api.nal.usda.gov/fdc/v1"
    CARB_NUTRIENT_ID = 1005  # "Carbohydrate, by difference"
    PREFERRED_DATA_TYPES = ["Foundation", "SR Legacy"]
    SEARCH_PAGE_SIZE = 25  # mehr Kandidaten = bessere Auswahlbasis fürs Ranking

    # Begriffe, die auf verarbeitete/zusammengesetzte statt rohe/generische
    # Lebensmittel hindeuten. Senken den Score im Ranking.
    PROCESSED_KEYWORDS = {
        "pie", "croissant", "dehydrated", "powder", "dried", "juice", "sauce",
        "chips", "cooked", "canned", "candied", "syrup", "extract", "concentrate",
        "frozen", "cake", "muffin", "cereal", "jam", "jelly", "butter", "cider",
        "vinegar", "baked", "fried", "sweetened", "flavored", "drink", "beverage",
        "smoothie", "yogurt", "ice cream", "pudding", "bread", "roll", "pastry",
        "prepared", "fritter", "turnover", "strudel", "cobbler", "tart",
        "puree", "nectar", "cocktail", "baby food", "sorbet", "crisp", "dumpling",
    }

    def __init__(self, api_key: str = None):
        self.api_key = api_key or USDA_API_KEY
        self._cache: Dict[str, Tuple[float, str]] = {}

    def carbs_per_100g(self, food_name: str) -> Tuple[float, str]:
        """Gibt (kohlenhydrate_pro_100g, gefundener_usda_name) zurück."""
        key = food_name.strip().lower()
        if key in self._cache:
            return self._cache[key]

        best = self.best_match(food_name)
        carbs = self._extract_carbs(best)
        if carbs is None:
            raise USDALookupError(
                f"Kein Kohlenhydrat-Wert für '{best.get('description')}' gefunden"
            )

        result = (carbs, best.get("description", food_name))
        self._cache[key] = result
        return result

    def best_match(self, food_name: str) -> dict:
        """Sucht Kandidaten und gibt den nach Heuristik besten Treffer zurück."""
        foods = self._search(food_name, data_types=self.PREFERRED_DATA_TYPES)
        if not foods:
            foods = self._search(food_name, data_types=None)  # Fallback: auch Branded/Survey
        if not foods:
            raise USDALookupError(f"Keine USDA-Treffer für '{food_name}'")
        return max(foods, key=lambda f: self._score(f, food_name))

    def ranked_candidates(self, food_name: str, top_n: int = 5) -> list:
        """Debug-Hilfe: zeigt die Top-N Kandidaten inkl. Score, absteigend sortiert."""
        foods = self._search(food_name, data_types=self.PREFERRED_DATA_TYPES) or \
                self._search(food_name, data_types=None)
        scored = sorted(foods, key=lambda f: self._score(f, food_name), reverse=True)
        return [(f.get("description"), self._score(f, food_name)) for f in scored[:top_n]]

    def _search(self, query: str, data_types: Optional[List[str]]) -> list:
        params = {"api_key": self.api_key, "query": query, "pageSize": self.SEARCH_PAGE_SIZE}
        if data_types:
            params["dataType"] = ",".join(data_types)
        response = requests.get(f"{self.BASE_URL}/foods/search", params=params, timeout=10)
        response.raise_for_status()
        return response.json().get("foods", [])

    def _score(self, food: dict, query: str) -> tuple:
        """Höher = besser. Reihenfolge der Tupel-Elemente = Priorität der Kriterien."""
        description = food.get("description", "")
        desc_lower = description.lower()
        head = description.split(",")[0]  # USDA-Konvention: Basiswort vor dem ersten Komma

        starts_similar = self._similar_prefix(head, query)
        has_raw = "raw" in desc_lower
        processed_penalty = sum(1 for kw in self.PROCESSED_KEYWORDS if kw in desc_lower)
        data_type_rank = {"Foundation": 2, "SR Legacy": 1}.get(food.get("dataType"), 0)

        return (
            1 if starts_similar else 0,
            1 if has_raw else 0,
            -processed_penalty,
            data_type_rank,
            -len(description),  # kürzer = generischer, als Tie-Breaker
        )

    @staticmethod
    def _similar_prefix(description_head: str, query: str, chars: int = 5) -> bool:
        """Grobe Heuristik statt echter Pluralbildung: vergleicht die ersten
        `chars` Buchstaben. Deckt die meisten englischen Singular/Plural-Fälle ab
        ('apple'/'apples', 'peach'/'peaches', 'banana'/'bananas', ...).
        """
        a = re.sub(r"[^a-zA-Z]", "", description_head).lower()
        b = re.sub(r"[^a-zA-Z]", "", query).lower()
        n = min(chars, len(a), len(b))
        if n < 3:
            return a == b
        return a[:n] == b[:n]

    def _extract_carbs(self, food: dict) -> Optional[float]:
        for nutrient in food.get("foodNutrients", []):
            if nutrient.get("nutrientId") == self.CARB_NUTRIENT_ID:
                return nutrient.get("value")
        for nutrient in food.get("foodNutrients", []):
            if "carbohydrate" in nutrient.get("nutrientName", "").lower():
                return nutrient.get("value")
        return None

### Debug: Kandidaten-Ranking einsehen

Falls ein Ergebnis unplausibel wirkt, hier nachsehen, welche Kandidaten es gab und warum welcher gewonnen hat. Die Tupel-Werte entsprechen `(Namensähnlichkeit, hat "raw", -Straf-Wörter, Datentyp-Rang, -Länge)` — der erste Eintrag hat gewonnen.

In [6]:
repo = USDANutritionRepository()

for begriff in ["apple", "banana", "strawberries", "peach"]:
    print(f"--- {begriff} ---")
    for name, score in repo.ranked_candidates(begriff, top_n=5):
        print(f"  {score}  {name}")
    print()

--- apple ---
  (1, 1, 0, 2, -28)  Apples, fuji, with skin, raw
  (1, 1, 0, 2, -28)  Apples, gala, with skin, raw
  (1, 1, 0, 2, -36)  Apples, granny smith, with skin, raw
  (1, 1, 0, 1, -25)  Apples, raw, without skin
  (1, 1, 0, 1, -40)  Apples, raw, golden delicious, with skin

--- banana ---
  (1, 1, 0, 2, -22)  Bananas, overripe, raw
  (1, 1, 0, 2, -36)  Bananas, ripe and slightly ripe, raw
  (1, 1, 0, 1, -12)  Bananas, raw
  (1, 0, -2, 1, -37)  Bananas, dehydrated, or banana powder
  (0, 1, 0, 1, -19)  Pepper, banana, raw

--- strawberries ---
  (1, 1, 0, 2, -17)  Strawberries, raw
  (1, 1, 0, 1, -17)  Strawberries, raw
  (1, 1, -2, 1, -38)  Strawberry-flavor beverage mix, powder
  (1, 1, -2, 1, -39)  Strawberries, frozen, sweetened, sliced
  (0, 1, 0, 1, -20)  Toppings, strawberry

--- peach ---
  (1, 1, 0, 2, -20)  Peaches, yellow, raw
  (1, 1, 0, 1, -20)  Peaches, yellow, raw
  (1, 0, -1, 1, -47)  Peaches, canned, water pack, solids and liquids
  (1, 0, -1, 1, -50)  Peaches, d

### Dein Testfall

In [11]:
analyzer_test = MealAnalyzer(parser=IngredientParser(), repository=USDANutritionRepository())
m = analyzer_test.analyze("42 g apple, 55 g banana, 56 g strawberries, 65 g peach")
print(m.report())

Mahlzeit: 42 g apple, 55 g banana, 56 g strawberries, 65 g peach
-----------------------------------------------------------------
  • apple                  42.0 g →  6.6 g   (USDA: Apples, fuji, with skin, raw)
  • banana                 55.0 g →  11.1 g   (USDA: Bananas, overripe, raw)
  • strawberries           56.0 g →  4.5 g   (USDA: Strawberries, raw)
  • peach                  65.0 g →  6.6 g   (USDA: Peaches, yellow, raw)
-----------------------------------------------------------------
  Gesamt-Kohlenhydrate: 28.7 g


## 5. Domain Service: `MealAnalyzer`

Orchestriert Parser und Repository. Enthält selbst keine API- oder Regex-Details — genau das ist der Sinn der Trennung.

In [8]:
class MealAnalyzer:
    """Domain Service: baut aus Freitext ein vollständiges Meal-Aggregat."""

    def __init__(self, parser: IngredientParser, repository: USDANutritionRepository):
        self.parser = parser
        self.repository = repository

    def analyze(self, description: str) -> Meal:
        parsed_items = self.parser.parse(description)

        ingredients = []
        for name, grams in parsed_items:
            carbs_per_100g, matched_name = self.repository.carbs_per_100g(name)
            carbs = Carbohydrates(grams=grams * carbs_per_100g / 100.0)
            ingredients.append(
                Ingredient(
                    name=name,
                    matched_usda_name=matched_name,
                    weight_g=grams,
                    carbs=carbs,
                )
            )

        return Meal(description=description, ingredients=ingredients)

## 6. Verwendung

Dein Beispiel, jetzt mit expliziten Gramm-Angaben statt vagen Mengen:

In [ ]:
analyzer = MealAnalyzer(
    parser=IngredientParser(),
    repository=USDANutritionRepository(),
)

beschreibung = "150 g Haferflocken, 50 g Apfel, 60 g Banane, 75 g Pfirsich"

mahlzeit = analyzer.analyze(beschreibung)
print(mahlzeit.report())

### Nur die Zahl ausgeben

In [ ]:
print(f"Kohlenhydrate: {mahlzeit.total_carbs.grams:.1f} g")

## 7. Weitere Beispiele zum Ausprobieren

In [ ]:
beispiele = [
    "30 g Vollkornbrot, 10 g Butter, 15 g Erdbeermarmelade",
    "100 g Spaghetti roh, 120 g Tomatensoße",
    "150 g Naturjoghurt, 40 g Heidelbeeren, 10 g Honig",
]

for b in beispiele:
    m = analyzer.analyze(b)
    print(m.report())
    print()

## 8. Fehlerfälle

Zwei typische Fehlerquellen, sauber als eigene Exceptions modelliert:

In [ ]:
# Fehlendes Gewicht
try:
    analyzer.analyze("ein Apfel, 50 g Banane")
except IngredientParsingError as e:
    print(f"Parsing-Fehler: {e}")

# Unbekannte/kryptische Zutat
try:
    analyzer.analyze("50 g Xyzzyzutat123")
except USDALookupError as e:
    print(f"USDA-Fehler: {e}")

## 9. Erweiterungsideen

- **Andere Nährwerte** (Eiweiß = ID 1003, Fett = ID 1004, Energie = ID 1008) genauso über `USDANutritionRepository` abrufen — neues Value Object, gleiches Muster.
- **Bessere Treffer-Auswahl**: aktuell wird einfach der erste Suchtreffer genommen. Für Produktionscode lohnt sich Fuzzy-Matching (z. B. `rapidfuzz`) zwischen Zutatenname und `description`.
- **Deutsche Zutatennamen**: USDA ist eine US-Datenbank (englische Bezeichnungen). Für rein deutsche Eingaben ("Haferflocken" statt "oats") ggf. ein kleines Übersetzungs-Mapping vorschalten oder eine EU-Datenbank wie Open Food Facts nutzen.
- **Persistenter Cache**: `USDANutritionRepository._cache` ist aktuell nur In-Memory. Für wiederholte Nutzung z. B. in SQLite auslagern.
- **Async/Parallel**: bei vielen Zutaten mit `asyncio` + `aiohttp` parallelisieren statt sequenziell abzufragen.

## ⚠️ Hinweise

- USDA-Daten sind **pro 100 g des rohen/verarbeiteten Lebensmittels** — bei zubereiteten Speisen (z. B. gekochte Nudeln) können sich Gewicht und Nährwert stark vom Rohzustand unterscheiden. Ggf. `"Spaghetti, cooked"` statt `"Spaghetti"` suchen.
- Die deutschen Zutatennamen im Beispiel funktionieren nur, weil USDA teilweise auch gängige Begriffe erkennt — bei spezielleren Lebensmitteln lieber englische Suchbegriffe verwenden.
- `DEMO_KEY` ist stark ratenlimitiert (kein garantiertes Kontingent) — für mehr als ein paar Testaufrufe unbedingt einen eigenen kostenlosen Key holen.